In [ ]:
# ============================================================
# KoGPT2 전자제품 상담 챗봇
# Google Colab
# Trainer 사용 안함
# ============================================================

!pip -q install transformers sentencepiece accelerate

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW
from tqdm import tqdm
from difflib import SequenceMatcher

# ============================================================
# GPU 설정
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치 :", device)

# ============================================================
# 모델 로드
# ============================================================

model_name = "skt/kogpt2-base-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))
model.to(device)

# ============================================================
# 전자제품 상담 데이터
# ============================================================

base_examples = [
    {"q":"냉장고가 시원하지 않아","a":"냉장고 온도 설정을 확인하고 문이 제대로 닫히는지 점검해봐."},
    {"q":"냉장고에서 이상한 소리가 나","a":"냉장고가 수평으로 설치되어 있는지 확인하고 내부 물건이 진동하는지 살펴봐."},
    {"q":"에어컨이 시원하지 않아","a":"필터를 청소하고 냉방 모드가 맞는지 확인해봐. 냉매 부족일 수도 있어."},
    {"q":"에어컨에서 냄새가 나","a":"필터와 열교환기를 청소하고 송풍 모드로 충분히 건조시켜봐."},
    {"q":"세탁기가 탈수를 안 해","a":"세탁물이 한쪽으로 쏠렸는지 확인하고 배수 필터를 청소해봐."},
    {"q":"세탁기에서 물이 새","a":"급수 호스와 배수 호스 연결 상태를 점검해봐."},
    {"q":"TV 화면이 안 나와","a":"전원 케이블과 HDMI 연결 상태를 확인해봐."},
    {"q":"TV 리모컨이 작동하지 않아","a":"건전지를 교체하고 리모컨 수신부를 확인해봐."},
    {"q":"전자레인지가 작동하지 않아","a":"문이 제대로 닫혔는지 확인하고 콘센트 전원을 점검해봐."},
    {"q":"전자레인지에서 스파크가 튀어","a":"금속 용기가 들어있는지 확인하고 내부를 청소해봐."},
    {"q":"노트북 배터리가 빨리 닳아","a":"화면 밝기를 낮추고 백그라운드 프로그램을 정리해봐."},
    {"q":"노트북이 너무 뜨거워","a":"통풍구를 청소하고 평평한 곳에서 사용해봐."},
    {"q":"청소기 흡입력이 약해졌어","a":"먼지통을 비우고 필터를 청소해봐."},
    {"q":"청소기가 갑자기 꺼져","a":"과열 보호 기능이 작동했을 수 있으니 잠시 식힌 후 다시 사용해봐."},
    {"q":"가습기에서 물이 새","a":"물통 결합 상태와 패킹 손상 여부를 확인해봐."},
    {"q":"가습기에서 냄새가 나","a":"물통과 진동자를 세척하고 물을 자주 교체해줘."},
    {"q":"공기청정기 필터 교체 시기가 궁금해","a":"보통 6개월에서 1년 주기로 교체하지만 사용 환경에 따라 달라질 수 있어."},
    {"q":"공기청정기 바람이 약해","a":"필터 오염 상태를 확인하고 청소 또는 교체해봐."},
    {"q":"스마트폰 충전이 안 돼","a":"충전 케이블과 충전 단자를 확인하고 다른 충전기로 테스트해봐."},
    {"q":"스마트폰이 느려졌어","a":"사용하지 않는 앱을 종료하고 저장 공간을 확보한 뒤 재부팅해봐."}
]

# ============================================================
# 데이터 증강
# ============================================================

texts = []

for item in base_examples:

    texts.append(
        f"질문: {item['q']}\n답변: {item['a']}"
    )

    texts.append(
        f"질문: {item['q']}?\n답변: {item['a']}"
    )

    texts.append(
        f"질문: {item['q']}요\n답변: {item['a']}"
    )

texts = texts * 20

# ============================================================
# Dataset
# ============================================================

class ElectronicsDataset(Dataset):

    def __init__(self, texts):

        self.data = []

        for text in texts:

            ids = tokenizer.encode(
                text,
                truncation=True,
                max_length=128
            )

            self.data.append(torch.tensor(ids))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# ============================================================
# Collate
# ============================================================

def collate_fn(batch):

    max_len = max(len(x) for x in batch)

    input_ids = []

    for ids in batch:

        pad_len = max_len - len(ids)

        padded = torch.cat([
            ids,
            torch.full(
                (pad_len,),
                tokenizer.pad_token_id,
                dtype=torch.long
            )
        ])

        input_ids.append(padded)

    input_ids = torch.stack(input_ids)

    labels = input_ids.clone()

    return {
        "input_ids": input_ids,
        "labels": labels
    }

# ============================================================
# DataLoader
# ============================================================

dataset = ElectronicsDataset(texts)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

# ============================================================
# Optimizer
# ============================================================

optimizer = AdamW(
    model.parameters(),
    lr=5e-5
)

# ============================================================
# 학습
# ============================================================

epochs = 2

model.train()

for epoch in range(epochs):

    total_loss = 0

    progress = tqdm(loader)

    for batch in progress:

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        progress.set_description(
            f"Epoch {epoch+1}/{epochs}"
        )

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    print(
        f"Epoch {epoch+1} 평균 Loss : {total_loss/len(loader):.4f}"
    )

print("\n학습 완료!")

# ============================================================
# FAQ 검색
# ============================================================

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def find_best_answer(question):

    question = question.replace(" ", "")

    best_score = 0
    best_answer = None

    for item in base_examples:

        q = item["q"].replace(" ", "")

        score = similarity(question, q)

        keywords = [
            "에어컨",
            "냉장고",
            "세탁기",
            "TV",
            "전자레인지",
            "노트북",
            "청소기",
            "가습기",
            "공기청정기",
            "스마트폰"
        ]

        for keyword in keywords:

            if keyword in question and keyword in q:
                score += 0.3

        if score > best_score:
            best_score = score
            best_answer = item["a"]

    if best_score >= 0.35:
        return best_answer

    return None

# ============================================================
# 챗봇
# ============================================================

def chatbot(question):

    faq = find_best_answer(question)

    if faq:
        return faq

    model.eval()

    prompt = f"질문: {question}\n답변:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            num_beams=3,
            repetition_penalty=1.5,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    if "답변:" in text:
        answer = text.split("답변:")[-1].strip()
    else:
        answer = text.strip()

    return answer

# ============================================================
# 채팅
# ============================================================

print("\n==============================")
print("전자제품 상담 챗봇")
print("종료 : quit")
print("==============================")

while True:

    q = input("\n사용자 : ")

    if q.lower() == "quit":
        break

    print("챗봇 :", chatbot(q))

사용 장치 : cuda


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Epoch 1/2: 100%|██████████| 150/150 [00:24<00:00,  6.15it/s, loss=1.0744]


Epoch 1 평균 Loss : 2.3154


Epoch 2/2: 100%|██████████| 150/150 [00:24<00:00,  6.09it/s, loss=0.3791]


Epoch 2 평균 Loss : 0.7299

학습 완료!

전자제품 상담 챗봇
종료 : quit

사용자 : 에어컨
챗봇 : 필터와 열교환기를 청소하고 송풍 모드로 충분히 건조시켜봐.

사용자 : 에어컨 문제 있어
챗봇 : 필터와 열교환기를 청소하고 송풍 모드로 충분히 건조시켜봐.

사용자 : 스마트폰 문제있어
챗봇 : 사용하지 않는 앱을 종료하고 저장 공간을 확보한 뒤 재부팅해봐.

사용자 : 충전이 안 돼
챗봇 : 충전 케이블과 충전 단자를 확인하고 다른 충전기로 테스트해봐.

사용자 : 세탁기에서 물새
챗봇 : 급수 호스와 배수 호스 연결 상태를 점검해봐.

사용자 : 에어컨 찬 바람 안 나와
챗봇 : 필터와 열교환기를 청소하고 송풍 모드로 충분히 건조시켜봐.
